# Code Verification by the Method of Manufactured Solutions

This notebook presents the Method of Manufactured Solutions (MMS) used for code verification and provides a walkthrough of the code that implements it.

Verification and Validation answer two distinct questions [[2]](#ref2)[[6]](#ref6): 

- **Verification** Does the code solve its equations correctly, at the order of accuracy the discretisation claims?
- **Validation**  Are these the right equations for the physical problem?

In short, verification is *solving the equations right*, and validation is *solving the right equations* [[7]](#ref7).

Verifying a code means comparing its solution against an exact one. The problem with classical exact solutions is that they exist
only for very simple geometries, loads and material laws, and even then they leave terms of the
equations untested [[2]](#ref2)[[4]](#ref4). A code can pass them and still be wrong. 
MMS removes this limitation: it builds the problem around a solution we construct, so an exact
solution is always available.

## The MMS in a nutshell

Suppose we wish to solve the mathematical model $\mathcal{L}(u) = 0$, where $\mathcal{L}$ is the
differential operator and $u$ the unknown. The main idea is to choose a manufactured solution
$\hat{u}$ first, then apply the governing equations to it: what remains, $s = \mathcal{L}(\hat{u})$,
is the source term. By construction, $\hat{u}$ is the exact solution of the modified model
[[1]](#ref1)[[2]](#ref2)[[5]](#ref5)

$$\mathcal{L}(u) = s$$

The boundary conditions are derived from $\hat{u}$ as well. Three properties make this practical
[[5]](#ref5):

- $s$ depends on the coordinates only, not on $u$. It enters the code as a known load, that is, as
  input.
- Since $s$ is a known function, no discrete version of $\mathcal{L}$ is applied to it. It is
  only integrated into nodal loads, by quadrature.
- The left-hand side $\mathcal{L}(u)$ is unchanged, so the discretisation under test is not
  modified; only its input is.

In short, the expectation is that solving the modified problem gives back $\hat{u}$.

In practice, the code does not solve $\mathcal{L}(u) = s$ itself but a discrete version of it: a
finite element code solves its weak form on a mesh. Its solution $u_h$ is then only an
approximation of $\hat{u}$, and the difference $e = u_h - \hat{u}$ is the error. Because $\hat{u}$
is known exactly, this error can be computed rather than estimated.

### Measuring the error

Numerical error has three sources [[3]](#ref3): the discretisation, finite-precision arithmetic
and the tolerance the solvers stop at. A coding mistake adds a fourth contribution, the one the
study is looking for:

$$e = \varepsilon_h + \varepsilon_\text{ro} + \varepsilon_\text{it} + \varepsilon_\text{bug}$$

- $\varepsilon_h$, the *discretisation error*, which comes from the mesh and vanishes as it is
  refined;
- $\varepsilon_\text{ro}$, the *round-off error*, from finite-precision arithmetic;
- $\varepsilon_\text{it}$, the *iterative error*, from stopping the linear or Newton solver at a
  finite tolerance;
- $\varepsilon_\text{bug}$, the error caused by *coding mistakes*.

Under a correct implementation $\varepsilon_\text{bug} = 0$, and only $\varepsilon_h$ should remain,
provided $\varepsilon_\text{ro}$ and $\varepsilon_\text{it}$ are kept far below it.

The error is a field over $\Omega$, so its size is measured with norms integrated over the domain:

$$\|e\|_{L^2} = \left(\int_\Omega |e|^2\,d\Omega\right)^{1/2}, \qquad
  |e|_{H^1} = \left(\int_\Omega \|\nabla e\|^2\,d\Omega\right)^{1/2}$$

The $L^2$ norm measures the error itself, the $H^1$ semi-norm its gradient.
This suite also compares the strain energy SOFA computes for $u_h$ with the exact one, $E(\hat{u})$, through $\sqrt{2\,|E(\hat{u}) - E(u_h)|}$.

Order verification assumes that the discretisation error can be expanded in powers of the mesh size
$h$ [[5]](#ref5):

$$\|\varepsilon_h\| = C\,h^p + O(h^{p+1})$$

$p$ is the formal order of accuracy of the discretisation. $C$ does not depend on $h$, but on the
higher derivatives of the manufactured solution [[5]](#ref5). As $h$ decreases, the higher-order
terms vanish faster than the first one, so on fine enough meshes the leading term dominates and
$\|\varepsilon_h\| \approx C\,h^p$. Such meshes are in the *asymptotic regime*. On coarser ones, in the pre-asymptotic regime, the higher-order terms still weigh, and the observed order can differ from $p$ even for a correct code. For elements whose shape functions are polynomials of degree
$q$, $p = q + 1$ in the $L^2$ norm and $p = q$ in the $H^1$ semi-norm [[8]](#ref8).

$\varepsilon_\text{ro}$ and $\varepsilon_\text{it}$ do not follow $h$: the iterative error is set
by the solver tolerance, and round-off grows as the mesh is refined and the system becomes larger
and worse conditioned. They set a noise floor, and an error below it must not be used to measure
the order: the ratio of two such errors reflects round-off and solver tolerance, not the
discretisation. Both must stay well below $\varepsilon_h$ on the finest mesh [[5]](#ref5). A coding
mistake shows up as an error that does not vanish, or that vanishes at the wrong rate
[[4]](#ref4).

The code is judged by the rate at which the error decreases, not by its size. A single error value
on a single mesh does not tell whether the code is correct: how small is small enough depends on the
solution, the units and the mesh, all of which enter $C$, while $p$ depends on the discretisation
alone. The mesh is therefore refined by a factor $r = h_\text{coarse}/h_\text{fine} > 1$, and the
ratio of the two errors gives the observed order of accuracy [[5]](#ref5):

$$\hat{p} = \frac{\ln\!\left(\|e_{rh}\| / \|e_{h}\|\right)}{\ln r}$$

The test passes when $\hat{p}$ matches $p$. Checking the order is a much stronger criterion than
checking that the error is small, or even that it decreases [[4]](#ref4).

Everything under test is implemented in SOFA, in C++: the force fields
(`LinearSmallStrainFEMForceField`, `CorotationalFEMForceField`), the
solvers, and the components that turn a body force into nodal loads (`VectorSourceTerm`,
`FEMSourceTermIntegrator`). The suite does not modify any of them; they are the discrete
$\mathcal{L}$ whose left-hand side stays unchanged.

The Python scripts do everything around them:

- derive the source term $s$ and the boundary conditions from $\hat{u}$ and the material law;
- apply them to a SOFA scene as loads and constraints;
- solve the scene on a sequence of refined meshes, measure the error on each and read its order.

### Further reading

- Steinberg and Roache [[1]](#ref1): the original presentation of the method.
- Roache [[2]](#ref2): a short review of MMS for code verification.
- Roy [[3]](#ref3): a review of code and solution verification, with guidance on choosing manufactured solutions.
- Salari and Knupp [[4]](#ref4): the most complete treatment, including examples of coding mistakes detected by MSS
- Oberkampf and Roy [[5]](#ref5), Chapters 5 and 6: the textbook reference on code verification and MMS.
- Aycock *et al.* [[9]](#ref9): MMS applied to elastostatic solid mechanics in a finite element solver.


## Example: a trigonometric manufactured solution on a 2D square

The rest of this notebook walks through one verification study, building each piece of it in turn.

### The governing equations

Under small-strain deformations, the displacement $u$ of a linear elastic body in static equilibrium
and without external loads satisfies

$$\mathcal{L}(u) = -\nabla\cdot\sigma(u) = 0, \qquad
  \sigma = \lambda\,\mathrm{tr}(\varepsilon)\,I + 2\mu\,\varepsilon, \qquad
  \varepsilon = \tfrac{1}{2}\left(\nabla u + \nabla u^T\right)$$

This is the equation solved by the `LinearSmallStrainFEMForceField` component in SOFA. $\mu$ and
$\lambda$ are the Lamé parameters, obtained from Young's modulus $E$ and Poisson's ratio $\nu$.

The body $\Omega$ is a rectangle of length $L$ and width $W$, here a square with $L = W = 1$. Its
boundary is made of four sides: `left` ($x = 0$), `right` ($x = L$), `bottom` ($y = 0$) and `top`
($y = W$).

### The manufactured solution

We choose one sine period along each axis:

$$\hat{u}(x, y) = A \begin{bmatrix} \sin(kx)\cos(ky) \\ \cos(kx)\sin(ky) \end{bmatrix},
  \qquad k = \frac{2\pi}{L} = \frac{2\pi}{W}$$

$\hat{u}$ is not a physical solution and does not need to be one. It only has to be smooth, to leave
no derivative appearing in the equations identically zero, and to have derivatives small enough that
the asymptotic regime is reached on affordable meshes [[4]](#ref4)[[5]](#ref5). The amplitude $A$
sets the last of these.

Its gradient

$$\hat{\varepsilon} = \nabla\hat{u} = A k \begin{bmatrix} c & -s \\ -s & c \end{bmatrix},
  \qquad c = \cos(kx)\cos(ky), \quad s = \sin(kx)\sin(ky)$$

and the stress

$$\hat{\sigma} = 2 A k \begin{bmatrix} (\lambda+\mu)\,c & -\mu\,s \\ -\mu\,s & (\lambda+\mu)\,c \end{bmatrix}$$

Applying $\mathcal{L}$ to $\hat{u}$ gives the source term:

$$s = \mathcal{L}(\hat{u}) = -\nabla\cdot\hat{\sigma} = 2k^2(\lambda + 2\mu)\,\hat{u}$$


### The boundary conditions

The boundary conditions are taken from $\hat{u}$ as well. Multiplying $\mathcal{L}(u) = s$ by a test
function $v$, integrating over $\Omega$ and integrating by parts gives the weak form

$$\int_\Omega \sigma : \nabla v \, d\Omega - \int_{\partial\Omega} t \cdot v \, d\Gamma
  = \int_\Omega s \cdot v \, d\Omega, \qquad t = \sigma\,n$$

where $n$ is the outward normal and $t$ the traction. On each side, and in each direction, a
boundary condition prescribes one of two quantities:

- the displacement, $u = \hat{u}$ (Dirichlet). The test function vanishes there, and so does the
  boundary term;
- the traction, $t = \hat{t} = \hat{\sigma}\,n$ (Neumann), which enters the boundary term as a known
  load.

Since $\hat{u}$ is known everywhere, both conditions are consistent with it, and which one goes where
is a choice. Here, the displacement is prescribed on the whole boundary. It is not a zero Dirichlet condition, for instance on `left` it is $\hat{u}(0, y) = (0,\ A\sin(ky))$.

The modified problem is then

$$-\nabla\cdot\sigma(u) = 2k^2(\lambda + 2\mu)\,\hat{u} \quad \text{in } \Omega, \qquad
  u = \hat{u} \quad \text{on } \partial\Omega$$

and $\hat{u}$ is its exact solution. In this example $A = 0.02$, $E = 10^6$ and $\nu = 0.3$.

### From the problem to the code

Setting up an MMS study is systematic, so the problem above splits into separate concepts, each
with its own place in the repository:

```
Verification-Validation/
├── common/                   code shared by the suites
│   ├── geometry/             the domains Ω and their named boundary regions
│   └── sofa/
│       ├── prefabs/          how SOFA builds an elastic body on each geometry
│       ├── controllers.py    SOFA controllers the scenes need, e.g. the prescribed displacement clamp
│       └── scene.py          the base of every SOFA scene
└── verification/
    ├── manufactured/         modules that implement manufactured solutions and material laws; combining them forms the manufactured problem to be solved
    ├── scene/                the SOFA scene that loads a body with a manufactured problem
    ├── study/                the error measurement and the convergence study
    ├── case.py               reads a case: the study's configuration
    └── run.py                runs a case
```

Each of these is introduced below, where the walkthrough reaches it.

### The input file

A study is set up by an input file, a JSON file the suite reads into a dictionary. The walkthrough
builds it up as it goes, one key at a time. The problem above needs three keys:

- `geometry` names the geometry and gives its size;
- `solution` names the manufactured solution and gives its amplitude $A$.
- `material` names the constitutive law and gives its parameters.

Each `type` in the input file names a class.

In [ ]:
input_file = {
    "geometry": {"type": "Beam2D", "length": 1.0, "width": 1.0},
    "solution": {"type": "Trigonometric", "amplitude": 0.02},
    "material": {"type": "LinearElastic", "youngModulus": 1e6, "poissonRatio": 0.3},
}

### The manufactured problem

The geometry is built first, because the solution depends on it.

In [ ]:
import sys

# Make the repository's packages importable from this folder.
sys.path.insert(0, "../..")

from verification.case import GEOMETRIES

geometry_parameters = dict(input_file["geometry"])
geometry = GEOMETRIES[geometry_parameters.pop("type")](**geometry_parameters)

The solution's class is looked up by the geometry's dimension: `Trigonometric` in 2D is `Trigonometric2D` (`verification/manufactured/grid.py`). Solutions take the geometry as inputs because various parameters, such as $k$, might depend on the dimensions. Also, the displacement is prescribed on the geometry's regions.

In [ ]:
from verification.case import MANUFACTURED_SOLUTIONS

solution = MANUFACTURED_SOLUTIONS[(geometry.dim, input_file["solution"]["type"])](input_file["solution"], geometry)

The material converts the parameters of the input file into the ones its law is written with, which may depend on the number of spatial dimensions. `LinearElastic` (`verification/manufactured/materials.py`) converts $E$ and $\nu$ to the Lamé parameters $\mu$ and $\lambda$; hyperelastic materials return their own.

In [ ]:
from verification.case import MATERIALS

Material = MATERIALS[input_file["material"]["type"]]
parameters = Material.get_material_parameters(input_file["material"], geometry.spatial_dimensions)
material = Material(*parameters)

The solution and the material are paired into the manufactured problem, which differentiates $\hat{u}$ symbolically, with `sympy`, into the stress $\hat{\sigma}$ and the source term $s$. The problem is `ManufacturedProblem` (`verification/manufactured/problem.py`). It also compiles each expression into a numpy function evaluated at any array of points: `problem.u`, `problem.grad_u`, `problem.stress` and `problem.source`. These work as inputs to apply source terms such as body forces, prescribe displacement on boundaries and traction BCs. Additionally, the error is measured against them during the convergence analysis. These are the expressions it derived, with the numbers of the input file substituted.

In [ ]:
import sympy as sp
from IPython.display import Math, display

from verification.manufactured import ManufacturedProblem

problem = ManufacturedProblem(solution, material, geometry.spatial_dimensions)


def rounded(expression):
    """The expression with its floating-point numbers cut to 6 significant digits, for display."""
    return expression.xreplace({f: sp.Float(f, 6) for f in expression.atoms(sp.Float)})


# The symbolic expressions the problem derived.
display(Math(r"\hat{u} = " + sp.latex(rounded(problem.displacement_expression))),
        Math(r"\hat{\sigma} = " + sp.latex(rounded(problem.stress_expression))),
        Math(r"s = " + sp.latex(rounded(problem.source_expression))))

The plot below uses `problem.u` to draw the exact $\hat{u}$ over the square. On the right, the grid of the undeformed square is displaced by $\hat{u}$: the boundary keeps its shape, but its points slide along it. The prescribed displacement is not zero on the boundary.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# û is small next to the square; magnify it so the deformation is visible.
MAGNIFICATION = 5

# Evaluate the compiled û on a fine grid of points covering the square.
x, y = np.meshgrid(np.linspace(0, geometry.length, 201), np.linspace(0, geometry.width, 201))
u = problem.u(np.stack([x, y], axis=-1))

fig, (magnitude, deformed) = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")

# Left: the magnitude of û as colour, its direction as arrows on a coarser grid.
field = magnitude.pcolormesh(x, y, np.linalg.norm(u, axis=-1), shading="gouraud")
fig.colorbar(field, ax=magnitude, label=r"$|\hat{u}|$")
step = slice(None, None, 20)
magnitude.quiver(x[step, step], y[step, step], u[step, step, 0], u[step, step, 1], color="white")
magnitude.set_title(r"$\hat{u}$: magnitude and direction")

# Right: each line of the undeformed grid, and the same line moved by the magnified û.
# The boundary lines are drawn thicker.
t = np.linspace(0, 1, 201)
for c in np.linspace(0, 1, 11):
    for line in (np.stack([c * geometry.length + 0 * t, t * geometry.width], axis=-1),
                 np.stack([t * geometry.length, c * geometry.width + 0 * t], axis=-1)):
        moved = line + MAGNIFICATION * problem.u(line)
        deformed.plot(*line.T, color="0.8", linewidth=0.6)
        deformed.plot(*moved.T, color="black", linewidth=1.6 if c in (0, 1) else 0.8)
deformed.set_title(rf"Undeformed grid (grey) displaced by $\hat{{u}}$, magnified ×{MAGNIFICATION}")

for ax in (magnitude, deformed):
    ax.set_aspect("equal")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

### Solving on one mesh

SOFA now solves the modified problem on a mesh. This needs four more keys for the `input_file`:

- `forceField` is the SOFA component under test. It is the discrete $\mathcal{L}$, the operator whose order of accuracy the study verifies;
- `element` is the kind of element the geometry is meshed with, here `quad`. SOFA builds the topology container from it;
- `solvers` sets how SOFA solves the discrete system;
- `sourceQuadratureDegree` is the degree of the quadrature that integrates the source term $s$ into nodal loads.

In [ ]:
input_file["forceField"] = {"type": "LinearSmallStrainFEMForceField"}
input_file["element"] = "quad"
input_file["solvers"] = {
    "integration":  {"type": "StaticSolver"},
    "newton":       {"type": "NewtonRaphsonSolver", "maxNbIterationsNewton": 1,
                     "relativeInitialStoppingThreshold": 1e-6,
                     "relativeSuccessiveStoppingThreshold": 0,
                     "absoluteResidualStoppingThreshold": 0,
                     "printLog": False},
    "linearSolver": {"type": "SparseLDLSolver", "template": "CompressedRowSparseMatrixd"},
}
input_file["sourceQuadratureDegree"] = 3

The scene is built in three steps:

1. **The body.** A geometry knows nothing of SOFA; its prefab holds the instructions to build it in SOFA. `ElasticBeam` (`common/sofa/prefabs/grid.py`), the prefab of `Beam2D`, meshes the square with a regular grid of `quad` elements and adds the force field and the solvers.
2. **The source term.** `ApplyManufacturedSourceTerm` (`verification/scene/controllers.py`) evaluates `problem.source`, the $s$ derived above, at the nodes, and integrates it against the shape functions into nodal loads.
3. **The boundary conditions.** `RegionClamp` (`common/sofa/controllers.py`) adds one constraint per region where the solution prescribes the displacement, `problem.prescribe_displacement_on`, here all four sides. It moves their nodes to `problem.u`, the exact $\hat{u}$, and fixes them there.

In [ ]:
import Sofa.Core
import Sofa.Simulation

from common.sofa.conventions import VEC_DIM
from common.sofa.controllers import RegionClamp
from common.sofa.prefabs import PREFAB_BY_GEOMETRY
from verification.scene.controllers import ApplyManufacturedSourceTerm
from verification.scene.scene import MMSScene

vec_type = VEC_DIM[geometry.spatial_dimensions]


def solve(cells):
    """Build the scene for a mesh of `cells` cells per side and solve it. Returns its root and body."""
    root = Sofa.Core.Node("root")
    root.addObject("RequiredPlugin", pluginName=MMSScene.plugins)
    root.addObject("DefaultAnimationLoop")

    # 1.
    Prefab = PREFAB_BY_GEOMETRY[type(geometry)]
    prefab = root.addChild(Prefab(
        name=Prefab.__name__, geometry=geometry, grid_resolution=[count + 1 for count in cells],
        element=input_file["element"],
        configs={"material": input_file["material"], "forceField": input_file["forceField"],
                 "solvers": input_file["solvers"]}))
    body = prefab.mechanical_node()

    # 2.
    body.addObject(ApplyManufacturedSourceTerm(
        node=body, dofs=body.dofs, source=problem.source, vec_type=vec_type,
        element=input_file["element"], source_quadrature_degree=input_file["sourceQuadratureDegree"],
        name="bodyForceCtrl"))

    # 3.
    body.addObject(RegionClamp(
        geometry=geometry, dofs=body.dofs, node=body,
        prescribe_displacement_on=problem.prescribe_displacement_on,
        vec_type=vec_type, displacement=problem.u, name="clampCtrl"))

    # Initialise the scene, which runs the controllers, then solve it: one step of the static solver.
    Sofa.Simulation.init(root)
    Sofa.Simulation.animate(root, root.dt.value)
    return root, body


# The mesh: 8 cells along each side of the square.
cells = [8, 8]
root, body = solve(cells)

Steps 1 to 3, building the scene for one mesh, are encapsulated in `MMSScene` (`verification/scene/scene.py`). It only builds the scene, initialization and solution are performed separately by the class that handles the mesh convergence study, introduced below. A side can also be given a traction instead of a displacement; the cases in `verification/cases/` show how.

The plot draws SOFA's solution $u_h$, read off the nodes, over the exact $\hat{u}$, both magnified as before.

In [ ]:
from matplotlib.collections import PolyCollection

# u_h at the nodes: where SOFA moved them, minus where they started.
nodes = body.dofs.rest_position.array()
u_h = body.dofs.position.array() - nodes
quads = body.topology.quads.array()

fig, ax = plt.subplots(figsize=(5.5, 5.5), layout="constrained")

# The exact û, along the lines of the mesh.
for c in np.linspace(0, 1, cells[0] + 1):
    for line in (np.stack([c * geometry.length + 0 * t, t * geometry.width], axis=-1),
                 np.stack([t * geometry.length, c * geometry.width + 0 * t], axis=-1)):
        moved = line + MAGNIFICATION * problem.u(line)
        ax.plot(*moved.T, color="0.6", linewidth=0.8)

# SOFA's solution u_h.
ax.add_collection(PolyCollection((nodes + MAGNIFICATION * u_h)[quads],
                                 facecolor="none", edgecolor="tab:red", linewidth=1.2))

ax.set_aspect("equal")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title(rf"$u_h$ on the {cells[0]}×{cells[1]} mesh (red) and $\hat{{u}}$ (grey), magnified ×{MAGNIFICATION}")

### Measuring the error

The norms $\|e\|_{L^2}$ and $|e|_{H^1}$ are integrated by quadrature over each element, with `MeshQuadrature` (`verification/study/quadrature.py`). At every quadrature point, $u_h$ and its gradient are interpolated from the nodes with the element's shape functions, whereas $\hat{u}$ and $\nabla\hat{u}$ are evaluated exactly. One more key sets the quadrature: `errorQuadratureDegree`.

In [ ]:
from common.sofa.conventions import ELEMENTS
from verification.study.quadrature import MeshQuadrature

input_file["errorQuadratureDegree"] = 2


def error_norms(body):
    """‖e‖_L² and |e|_H¹ of the solution SOFA found on `body`'s mesh."""
    nodes = body.dofs.rest_position.array()
    u_h = body.dofs.position.array() - nodes

    # The quadrature points and weights of every element of the mesh.
    quadrature = MeshQuadrature(nodes, body.topology.quads.array(),
                                ELEMENTS[input_file["element"]].cpp,
                                input_file["errorQuadratureDegree"])

    # The error and its gradient at every quadrature point: u_h interpolated, û evaluated exactly.
    e = quadrature.values(u_h) - quadrature.sample(problem.u)
    grad_e = quadrature.grads(u_h) - quadrature.sample(problem.grad_u)

    # ‖e‖_L² = √(∫ |e|² dΩ) and |e|_H¹ = √(∫ ‖∇e‖² dΩ).
    return (np.sqrt(quadrature.integrate(np.sum(e ** 2, axis=-1))),
            np.sqrt(quadrature.integrate(np.sum(grad_e ** 2, axis=(-2, -1)))))


L2, H1 = error_norms(body)
print(f"‖e‖_L2 = {L2:.3e}, |e|_H1 = {H1:.3e}")

These two numbers alone do not say whether the code is correct. That is decided by how they decrease as the mesh is refined. The computation of the errors is encapsulated in `Measurement`, and each norm in a metric class: `L2`, `H1` and `EnergyDifference`. `METRICS` lists the metrics every mesh is measured with. All are in `verification/study/metrics.py`.

### Refining the mesh

The scene is now solved on a sequence of meshes, each $r$ times finer than the one before, and the
observed order $\hat{p}$ is computed between each mesh and the one before it. The shape functions of `quad` elements
have degree $q = 1$, so they should give $p = 2$ in $L^2$ and $p = 1$ in $H^1$, and $p = 1$ for `EnergyDifference`. Three keys set this:

- `mesh` gives the coarsest mesh's cells per side, the number of meshes, and $r$;
- `expectedOrder` gives $p$ for each metric;
- `expectedOrderTolerance` is how far $\hat{p}$ on the finest pair of meshes may sit from $p$.

In [ ]:
# 3×3 cells, then 6×6, 12×12, 24×24 and 48×48.
input_file["mesh"] = {"cells": [3, 3], "levels": 5, "refinementRatio": 2}
input_file["expectedOrder"] = {"L2": 2.0, "H1": 1.0, "EnergyDifference": 1.0}
input_file["expectedOrderTolerance"] = 0.15

mesh = input_file["mesh"]
spacings, norms = [], []
for level in range(mesh["levels"]):
    level_cells = [count * mesh["refinementRatio"] ** level for count in mesh["cells"]]
    level_root, level_body = solve(level_cells)
    spacings.append(geometry.length / level_cells[0])
    norms.append(error_norms(level_body))
    # Free this mesh's scene before building the next one.
    Sofa.Simulation.unload(level_root)

# One row per mesh, one column per norm: ‖e‖_L² and |e|_H¹.
norms = np.array(norms)
# p̂ = ln(‖e_rh‖ / ‖e_h‖) / ln r, between each mesh and the one before it.
orders = np.log(norms[:-1] / norms[1:]) / np.log(mesh["refinementRatio"])

print(f"{'h':>8} {'L2':>10} {'p̂':>6} {'H1':>10} {'p̂':>6}")
for level, h in enumerate(spacings):
    rates = orders[level - 1] if level else (np.nan, np.nan)
    print(f"{h:8.4f} {norms[level, 0]:10.3e} {rates[0]:6.2f} {norms[level, 1]:10.3e} {rates[1]:6.2f}")

# The test: p̂ on the finest pair of meshes matches p, within the tolerance.
for name, order in zip(("L2", "H1"), orders[-1]):
    expected = input_file["expectedOrder"][name]
    passed = abs(order - expected) <= input_file["expectedOrderTolerance"]
    print(f"{name}: p̂ = {order:.2f}, p = {expected}: {'pass' if passed else 'fail'}")

This loop is encapsulated in `ErrorConvergenceStudy` (`verification/study/study.py`). It judges the orders more strictly than the single check above:

- a rate is discarded when either error is below the noise floor, `relativeNoiseFloor` times the size of the exact field. Below it, the round-off and iterative errors $\varepsilon_\text{ro}$ and $\varepsilon_\text{it}$ from the start dominate, and a rate measured there is meaningless;
- a rate is also discarded when the errors oscillate or diverge;
- the order is the one on the finest meshes, and is kept only once at least two consecutive rates agree within `asymptoticTolerance`: the meshes are then in the asymptotic regime, and rates from coarser meshes are discarded;
- the study fails if a Newton solve did not converge.

In [ ]:
input_file["relativeNoiseFloor"] = 1e-12
input_file["asymptoticTolerance"] = 0.10

## Putting it together

The input file is now complete. This is all a study needs, as the JSON file read by `run.py`:

```json
{
  "geometry":    { "type": "Beam2D", "length": 1.0, "width": 1.0 },
  "element":     "quad",
  "material":    { "type": "LinearElastic", "youngModulus": 1e6, "poissonRatio": 0.3 },
  "forceField":  { "type": "LinearSmallStrainFEMForceField" },
  "solution":    { "type": "Trigonometric", "amplitude": 0.02 },
  "errorQuadratureDegree": 3,
  "sourceQuadratureDegree": 3,
  "solvers": {
    "integration":  { "type": "StaticSolver" },
    "newton":       { "type": "NewtonRaphsonSolver", "maxNbIterationsNewton": 1,
                      "relativeInitialStoppingThreshold": 1e-6,
                      "relativeSuccessiveStoppingThreshold": 0,
                      "absoluteResidualStoppingThreshold": 0,
                      "printLog": false },
    "linearSolver": { "type": "SparseLDLSolver", "template": "CompressedRowSparseMatrixd" }
  },
  "mesh":        { "cells": [3, 3], "levels": 5, "refinementRatio": 2 },
  "asymptoticTolerance": 0.10,
  "expectedOrder": { "L2": 2.0, "H1": 1.0, "EnergyDifference": 1.0 },
  "expectedOrderTolerance": 0.15,
  "relativeNoiseFloor": 1e-12
}
```

Everything the walkthrough built by hand is done by two classes:

- `Case` (`verification/case.py`) checks the input file, then builds the geometry, the solution, the material and the manufactured problem from it;
- `ErrorConvergenceStudy` builds the scene of every mesh with `MMSScene` and solves it, then computes the errors of that mesh's solution $u_h$ with `Measurement` and their norms with the metrics listed in `METRICS`, and finally judges the order of each metric. It prints one row per mesh with the rate of each metric, or why it was discarded, then the order each metric settled on next to the expected one.

It also draws two figures: the error of each metric against $h$, its segments green where the rate was kept and red where it was discarded; and $|u_h|$ on each mesh, which approaches $|\hat{u}|$ as the mesh is refined.

In [ ]:
from verification.case import Case
from verification.study.study import ErrorConvergenceStudy

# The path only names the case; the input file is passed as the dictionary built above.
case = Case("LinearSmallStrainFEMForceField/Trigonometric2D_Quad_Dirichlet.json", input_file)
study = ErrorConvergenceStudy(case)
study.run()
print("\nverified:", study.verified)

# The convergence plot the study drew while running: the error of each metric against h.
display(study.plots["convergence"])

# |u_h| on each mesh, coarsest to finest, drawn on the displaced mesh.
study.plot_fields()
display(study.plots["fields"])

Saved as a JSON file under `verification/cases/<force field>/`, the input file is a *case*, and the
runner does all of the above from the command line, writing the results to `verification/results/`:

```bash
python verification/run.py LinearSmallStrainFEMForceField/Trigonometric2D_Quad.json
python verification/run.py --all     # every case, then one summary line per case
```

To inspect one mesh of a case in the SOFA GUI, the runner also acts as a SOFA scene file:

```bash
runSofa -l SofaPython3 verification/run.py --argv LinearSmallStrainFEMForceField/Trigonometric2D_Quad.json --argv 0
```

The first argument is the path to the .json case file and the second is the mesh refinement level to be built.

<a id="references"></a>
## References

<a id="ref1"></a>
**[1]** S. Steinberg and P. J. Roache, "Symbolic manipulation and computational fluid dynamics,"
*Journal of Computational Physics*, 57(2):251&ndash;284, 1985. See also P. J. Roache and
S. Steinberg, *AIAA Journal*, 22(10):1390&ndash;1394, 1984.

<a id="ref2"></a>
**[2]** P. J. Roache, "Code verification by the method of manufactured solutions,"
*Journal of Fluids Engineering*, 124(1):4&ndash;10, 2002.
[doi:10.1115/1.1436090](https://doi.org/10.1115/1.1436090)

<a id="ref3"></a>
**[3]** C. J. Roy, "Review of code and solution verification procedures for computational
simulation," *Journal of Computational Physics*, 205(1):131&ndash;156, 2005.
[doi:10.1016/j.jcp.2004.10.036](https://doi.org/10.1016/j.jcp.2004.10.036)

<a id="ref4"></a>
**[4]** K. Salari and P. Knupp, *Code Verification by the Method of Manufactured Solutions*,
SAND2000-1444, Sandia National Laboratories, Albuquerque, NM, June 2000.

<a id="ref5"></a>
**[5]** W. L. Oberkampf and C. J. Roy, *Verification and Validation in Scientific Computing*,
Cambridge University Press, 2010. ISBN 978-0-521-11360-1. &mdash; *Chapter 5 (code verification and
order verification) and Chapter 6, §6.3 (MMS).*

<a id="ref6"></a>
**[6]** W. L. Oberkampf and T. G. Trucano, "Verification and validation in computational fluid
dynamics," *Progress in Aerospace Sciences*, 38(3):209&ndash;272, 2002.
[doi:10.1016/S0376-0421(02)00005-2](https://doi.org/10.1016/S0376-0421%2802%2900005-2)

<a id="ref7"></a>
**[7]** B. W. Boehm, *Software Engineering Economics*, Prentice-Hall, Englewood Cliffs, NJ, 1981.

<a id="ref8"></a>
**[8]** P. G. Ciarlet, *The Finite Element Method for Elliptic Problems*, North-Holland, Amsterdam,
1978. Reprinted by SIAM, Classics in Applied Mathematics 40, 2002.

<a id="ref9"></a>
**[9]** K. I. Aycock, N. Rebelo and B. A. Craven, "Method of manufactured solutions code
verification of elastostatic solid mechanics problems in a commercial finite element solver,"
*Computers and Structures*, 229:106175, 2020.
[doi:10.1016/j.compstruc.2019.106175](https://doi.org/10.1016/j.compstruc.2019.106175)

<a id="ref10"></a>
**[10]** S. Étienne, A. Garon and D. Pelletier, "Some manufactured solutions for verification of
fluid-structure interaction codes," *Computers and Structures*, 106&ndash;107:56&ndash;67, 2012.
[doi:10.1016/j.compstruc.2012.04.006](https://doi.org/10.1016/j.compstruc.2012.04.006)